In [ ]:
!pip install faiss-cpu sentence_transformers pypdf

In [ ]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [ ]:
modle = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
resume_files = [
    ("Alaguraja_Resume.pdf", "Alaguraja K"),
    ("Santhosh_C_Resume.pdf", "Santhosh C"),
    ("Shivaji_Resume.pdf", "Shivaji S B")
]

In [ ]:
def extract_text(pdf_path):
  reader = PdfReader(pdf_path)

  text = ""
  for page in reader.pages:
    page_text = page.extract_text()

    if page_text:
      text += page_text + "\n"
  return text

In [ ]:
def chunk_text(text, chunk_size=500):
  chunks=[]
  for i in range(0,len(text),chunk_size):
    chunks.append(text[i:i+chunk_size])
  return chunks

In [ ]:
all_chunks = []
metadata = []
for pdf_file, candidate in resume_files:
  text = extract_text(pdf_file)
  chunks = chunk_text(text)

  for chunk in chunks:
    all_chunks.append(chunk)
    metadata.append({
        "candidate": candidate,
        "file": pdf_file
    })

In [ ]:
embeddings = modle.encode(all_chunks)

In [ ]:
embeddings.shape

(17, 384)

In [ ]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype("float32"))
print("Total chunks stored:", index.ntotal)

Total chunks stored: 17


In [ ]:
def search_resumes(top_k):
  query = input("Enter the query: ")
  query_embeddings = modle.encode([query])
  distances,indices = index.search(
      np.array(query_embeddings).astype("float32"),
      top_k
  )

  results = []

  for idx in indices[0]:

      results.append({
            "candidate": metadata[idx]["candidate"],
            "file": metadata[idx]["file"],
            "chunk": all_chunks[idx]
      })

  return results

In [ ]:
results = search_resumes(3)
results

Enter the query: Who knows python?


[{'candidate': 'Shivaji S B',
  'file': 'Shivaji_Resume.pdf',
  'chunk': 'SHIVAJI S B\n(+91) 99440 94104  |  sbshivaji2007@gmail.com  |  github.com/shiva-sb-kce  |  _.shiva_s.b._\nSUMMARY\nThird-year B.Tech Artificial Intelligence & Data Science student with hands-on experience in Python, Java, and web \ntechnologies. Passionate about software development and building practical solutions to real-world problems. Seeking an \nopportunity to contribute, learn, and grow within a forward-thinking organisation.\nTECHNICAL SKILLS\nProgramming Languages: Python, Java\nWeb Technologie'},
 {'candidate': 'Shivaji S B',
  'file': 'Shivaji_Resume.pdf',
  'chunk': 'ICATIONS\n• Participated in multiple hackathons, developing innovative solutions to real-world problems.\n• Completed three end-to-end projects, progressing from beginner to independent developer.\n• Earned a Python DSA Bootcamp Certification from Udemy, demonstrating strong problem-solving skills.\n• Solved 50+ LeetCode problems within a